# Gait feature and pipeline verification

**Purpose:** Check waveform, descriptor, PCA and participant-separation logic using synthetic data.

**Required inputs:** None; synthetic fixtures are generated in the notebook.

**Outputs:** Numerical and pipeline verification records.

**Usage:** Can be run independently. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports, fixed definitions, and implementation functions
from pathlib import Path
import csv
import json
import math
import platform
import sys
import zipfile

import numpy as np
import sklearn
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

OUT = Path('03_Bacek_Feature_Pipeline_Verification_results')
CHANNELS = (
    'hip_left', 'hip_right', 'knee_left', 'knee_right',
    'ankle_left', 'ankle_right',
    'pelvis_sagittal', 'pelvis_frontal', 'pelvis_transverse',
)
ALPHAS = (0.01, 0.1, 1, 10, 100, 1000)
PCA_COUNTS = (5, 10, 20, 40)
OUTER_FOLDS = {
    1: ('S08', 'S10', 'S16', 'S21'),
    2: ('S11', 'S13', 'S17', 'S20'),
    3: ('S01', 'S02', 'S05', 'S18'),
    4: ('S03', 'S07', 'S12', 'S14', 'S19'),
    5: ('S04', 'S06', 'S09', 'S15'),
}


def trial_mean_waveforms(channel_cycles):
    """Return nine trial-mean 101-point waveforms, in the fixed channel order."""
    if set(channel_cycles) != set(CHANNELS):
        raise ValueError('Nine exactly named gait-angle channels are required.')
    means = []
    n_cycles = None
    for name in CHANNELS:
        x = np.asarray(channel_cycles[name], dtype=np.float64)
        if x.ndim != 2 or x.shape[0] < 1 or x.shape[1] != 101:
            raise ValueError(f'Invalid (N, 101) array for {name}.')
        if not np.isfinite(x).all():
            raise ValueError(f'Nonfinite gait angle for {name}.')
        if n_cycles is None:
            n_cycles = x.shape[0]
        if x.shape[0] != n_cycles:
            raise ValueError('Channel cycle counts disagree.')
        means.append(x.mean(axis=0, dtype=np.float64))
    return np.stack(means)


def waveform_features(mean_waveforms):
    """W-ALL9; no centring or smoothing within a trial."""
    m = np.asarray(mean_waveforms, dtype=np.float64)
    if m.shape != (9, 101):
        raise ValueError('Expected nine 101-point trial-mean waveforms.')
    return m.reshape(909)


def descriptor_features(mean_waveforms):
    """D-ALL9: Q0..Q3, phase SD (ddof=0), ROM, repeated for each channel."""
    m = np.asarray(mean_waveforms, dtype=np.float64)
    if m.shape != (9, 101):
        raise ValueError('Expected nine 101-point trial-mean waveforms.')
    values = []
    for channel in m:
        values.extend(np.trapezoid(channel[q*25:q*25+26], dx=1.0) / 25.0
                      for q in range(4))
        values.append(np.std(channel, ddof=0))
        values.append(np.max(channel) - np.min(channel))
    return np.asarray(values, dtype=np.float64)


def descriptor_families(d):
    d = np.asarray(d, dtype=np.float64)
    if d.shape != (54,):
        raise ValueError('D-ALL9 must have 54 entries.')
    return {'D-FULL': d, 'D-PHASE': d.reshape(9, 6)[:, :4].ravel(),
            'D-SPREAD': d.reshape(9, 6)[:, 4:].ravel()}


def gait_pipeline(encoding, alpha=1.0, n_components=None):
    """The fixed scikit-learn W/D/P learner (training rows supplied at fit)."""
    if encoding in ('W-ALL9', 'D-ALL9'):
        if n_components is not None:
            raise ValueError('PCA count applies only to P-ALL9.')
        return Pipeline([('scale', StandardScaler()),
                         ('ridge', Ridge(alpha=alpha, fit_intercept=True))])
    if encoding == 'P-ALL9' and n_components in PCA_COUNTS:
        return Pipeline([('scale', StandardScaler()),
                         ('pca', PCA(n_components=n_components, whiten=False,
                                     random_state=2026)),
                         ('ridge', Ridge(alpha=alpha, fit_intercept=True))])
    raise ValueError('Unknown or invalid gait pipeline configuration.')


def pooled_rmse(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    if y.shape != yhat.shape or y.ndim != 1 or not np.isfinite(yhat).all():
        raise ValueError('Invalid matched prediction arrays.')
    return float(np.sqrt(np.mean((yhat - y)**2)))


def choose_configuration(candidates):
    """Exact-score tie: smaller input dimension, ID, then numerical settings."""
    return min(candidates, key=lambda c: (c['rmse'], c['n_features'],
                                         c['representation'], c.get('n_components') or 0,
                                         c['alpha']))


def assemble_contract(contract, blocks):
    """Explicit information whitelist; never include participant IDs or targets."""
    allowed = {
        'G': ('G',), 'C': ('C',), 'C+G': ('C', 'G'),
        'C+B': ('C', 'B'), 'C+B+G': ('C', 'B', 'G')
    }
    if contract not in allowed:
        raise ValueError('Unknown predictor contract.')
    return np.concatenate([np.asarray(blocks[key], dtype=np.float64).ravel()
                           for key in allowed[contract]])

results = []
inner_manifest = []

def check(name, function):
    try:
        function()
        results.append({'check': name, 'status': 'PASS', 'detail': ''})
        print('PASS:', name)
    except Exception as exc:
        results.append({'check': name, 'status': 'FAIL',
                        'detail': f'{type(exc).__name__}: {exc}'})
        print('FAIL:', name, '—', type(exc).__name__, exc)


## Synthetic feature tests


In [ ]:
# 2 — Check frozen trial means, feature order, and descriptors
phase = np.arange(101, dtype=np.float64)
synthetic_cycles = {
    name: np.stack([10*j + phase + 2, 10*j + phase - 2]).astype(np.float32)
    for j, name in enumerate(CHANNELS)
}


def test_trial_means_and_W():
    m = trial_mean_waveforms(synthetic_cycles)
    w = waveform_features(m)
    assert m.shape == (9, 101) and m.dtype == np.float64
    assert w.shape == (909,) and w.dtype == np.float64
    for j in range(9):
        np.testing.assert_allclose(w[j*101:(j+1)*101], phase + 10*j, atol=0, rtol=0)
    assert w[0] == 0 and w[100] == 100 and w[-1] == 180
    # The trial waveform retains its absolute angle offset.
    assert m[1, 0] == 10 and m[8, 0] == 80
    # A second trial cannot alter the first trial's mean.
    other = {k: v + 50 for k, v in synthetic_cycles.items()}
    np.testing.assert_array_equal(m, trial_mean_waveforms(synthetic_cycles))
    assert not np.array_equal(m, trial_mean_waveforms(other))


def test_float64_aggregation():
    sensitive = np.zeros((3, 101), dtype=np.float32)
    sensitive[:, 0] = [200_000_000, 0.5, -200_000_000]
    data = {name: sensitive.copy() for name in CHANNELS}
    result = trial_mean_waveforms(data)
    assert result.dtype == np.float64
    np.testing.assert_allclose(result[:, 0], 1.0 / 6.0, atol=1e-15, rtol=0)


def test_invalid_inputs():
    invalid = [
        {key: value for key, value in synthetic_cycles.items() if key != 'hip_left'},
        {**synthetic_cycles, 'hip_left': np.ones((2, 100))},
        {**synthetic_cycles, 'hip_left': np.ones((3, 101))},
        {**synthetic_cycles, 'hip_left': np.full((2, 101), np.nan)},
    ]
    for data in invalid:
        try:
            trial_mean_waveforms(data)
        except ValueError:
            continue
        raise AssertionError('Invalid gait input was accepted.')


def test_linear_descriptors():
    m = np.stack([10*j + 2*phase for j in range(9)])
    d = descriptor_features(m)
    assert d.shape == (54,) and d.dtype == np.float64
    expected_sd = 2 * np.sqrt(850)  # exact population variance of integers 0..100
    for j in range(9):
        expected = [10*j + 2*(25*q + 12.5) for q in range(4)]
        expected += [expected_sd, 200.0]
        np.testing.assert_allclose(d[6*j:6*j+6], expected, rtol=1e-13, atol=1e-12)


def test_quadratic_trapezoids():
    m = np.stack([(phase**2) + 10*j for j in range(9)])
    d = descriptor_features(m)
    for q in range(4):
        a, b = 25*q, 25*(q+1)
        # Analytic composite trapezoidal sum for x^2, step width 1:
        # (integral(x^2, a..b) + 25/6) / 25.
        expected = (b**3 - a**3) / 75 + 1/6
        np.testing.assert_allclose(d[q], expected, atol=1e-10, rtol=0)
    assert not np.isclose(d[0], np.mean(phase[:26]**2))


def test_mean_before_descriptors():
    a = np.zeros((2, 101), dtype=np.float32)
    a[0, 25], a[1, 75] = 100, 100
    data = {name: a.copy() for name in CHANNELS}
    d = descriptor_features(trial_mean_waveforms(data))
    assert d[5] == 50.0  # not mean of the individual-cycle maxima (100)
    assert d[4] > 0  # variation across phase of the mean, not cycle SD


def test_group_and_descriptor_family_order():
    m = trial_mean_waveforms(synthetic_cycles)
    w = waveform_features(m)
    d = descriptor_features(m)
    groups = {
        'hip': w[:202], 'knee': w[202:404],
        'ankle': w[404:606], 'pelvis': w[606:909], 'all': w,
    }
    assert [len(v) for v in groups.values()] == [202, 202, 202, 303, 909]
    f = descriptor_families(d)
    assert [f[k].shape for k in ('D-FULL', 'D-PHASE', 'D-SPREAD')] == [
        (54,), (36,), (18,)]
    np.testing.assert_array_equal(f['D-PHASE'][:4], d[:4])
    np.testing.assert_array_equal(f['D-PHASE'][4:8], d[6:10])
    np.testing.assert_array_equal(f['D-SPREAD'][:2], d[4:6])
    np.testing.assert_array_equal(f['D-SPREAD'][2:4], d[10:12])


check('Trial-mean waveform / W-ALL9 channel order and shape', test_trial_means_and_W)
check('Float64 mean accumulation of float32 cycle values', test_float64_aggregation)
check('Reject missing, malformed, inconsistent, and nonfinite gait arrays', test_invalid_inputs)
check('D-ALL9: linear waveform analytic descriptors and order', test_linear_descriptors)
check('D-ALL9: exact quadratic trapezoidal phase quartiles', test_quadratic_trapezoids)
check('Descriptors calculated after trial-level cycle averaging', test_mean_before_descriptors)
check('Anatomical waveform groups and descriptor-family order', test_group_and_descriptor_family_order)


## Participant-grouped fitting and scoring tests


In [ ]:
# 3 — Synthetic participant-grouped pipeline checks
rng = np.random.default_rng(2026)
participants = tuple(sorted({s for fold in OUTER_FOLDS.values() for s in fold}))
groups = np.repeat(participants, 8)
trial_ids = np.array([f'{s}_SYN{i+1:02d}' for s in participants for i in range(8)])
Xw = rng.normal(size=(len(groups), 909))
Xw += np.repeat(np.arange(len(participants))*0.05, 8)[:, None]
Xw[:, 0] = 7.0  # constant feature tests StandardScaler's convention
Xd = np.stack([descriptor_features(row.reshape(9, 101)) for row in Xw])
y = 0.4 + 0.02*Xd[:, 0] + rng.normal(scale=0.01, size=len(groups))


def test_candidates_and_exact_ties():
    assert len(ALPHAS) == 6 and len(PCA_COUNTS) == 4
    assert 6 + 6 + len(ALPHAS)*len(PCA_COUNTS) == 36
    candidates = [
        {'representation': r, 'n_features': d, 'alpha': a,
         'n_components': n, 'rmse': 0.5}
        for r, d, n in [('W-ALL9', 909, None), ('D-ALL9', 54, None),
                        ('P-ALL9', 5, 5)] for a in (0.1, 1.0)]
    best = choose_configuration(candidates)
    assert (best['representation'], best['alpha']) == ('P-ALL9', 0.1)
    candidates[-1]['rmse'] = 0.5 - 1e-12
    assert choose_configuration(candidates) is candidates[-1]  # no fuzzy tie


def test_outer_inner_group_boundaries():
    assert len(participants) == 21 and len(groups) == 168
    assert len(set(participants)) == 21
    assert sum(len(v) for v in OUTER_FOLDS.values()) == 21
    assert len(set(sum((list(v) for v in OUTER_FOLDS.values()), []))) == 21
    inner_manifest.clear()
    for outer_id, held_out in OUTER_FOLDS.items():
        is_test = np.isin(groups, held_out)
        dev_idx, test_idx = np.where(~is_test)[0], np.where(is_test)[0]
        assert set(groups[dev_idx]).isdisjoint(groups[test_idx])
        assert len(set(groups[dev_idx])) in (16, 17)
        seen = set()
        for inner_id, (a, b) in enumerate(
                GroupKFold(n_splits=4).split(Xw[dev_idx], y[dev_idx], groups[dev_idx]), 1):
            train_ids, val_ids = dev_idx[a], dev_idx[b]
            assert set(groups[train_ids]).isdisjoint(groups[val_ids])
            assert set(groups[test_idx]).isdisjoint(groups[train_ids])
            assert set(groups[test_idx]).isdisjoint(groups[val_ids])
            assert not (seen & set(val_ids))
            seen.update(map(int, val_ids))
            inner_manifest.append({
                'outer_fold': outer_id, 'inner_fold': inner_id,
                'train_participants': ';'.join(sorted(set(groups[train_ids]))),
                'validation_participants': ';'.join(sorted(set(groups[val_ids]))),
                'outer_test_participants': ';'.join(sorted(held_out)),
                'n_train_trials_synthetic': len(train_ids),
                'n_validation_trials_synthetic': len(val_ids),
            })
        assert seen == set(map(int, dev_idx))
    assert len(inner_manifest) == 20


def test_train_only_preprocessing_and_PCA():
    held_out = OUTER_FOLDS[1]
    train = ~np.isin(groups, held_out)
    test = ~train
    for X, name in [(Xw, 'W-ALL9'), (Xd, 'D-ALL9')]:
        pipe = gait_pipeline(name, alpha=1.0).fit(X[train], y[train])
        np.testing.assert_allclose(pipe['scale'].mean_, X[train].mean(axis=0),
                                   rtol=1e-12, atol=1e-12)
        assert pipe['scale'].n_samples_seen_ == int(train.sum())
        assert len(pipe.predict(X[test])) == int(test.sum())
    assert gait_pipeline('W-ALL9')['scale'].with_std
    # PCA is fitted only on outer-development trial rows; never on held-out people.
    for n in PCA_COUNTS:
        pipe = gait_pipeline('P-ALL9', alpha=1.0, n_components=n)
        pipe.fit(Xw[train], y[train])
        assert pipe['pca'].components_.shape == (n, 909)
        assert pipe['pca'].n_samples_ == int(train.sum())
        assert pipe['pca'].whiten is False
        np.testing.assert_allclose(pipe['scale'].mean_, Xw[train].mean(axis=0),
                                   rtol=1e-12, atol=1e-12)
        assert len(pipe.predict(Xw[test])) == int(test.sum())
        assert pipe['scale'].scale_[0] == 1.0  # constant-column convention
    # Within an inner fold, even inner-validation participants are excluded from PCA fitting.
    dev_idx = np.where(train)[0]
    a, b = next(GroupKFold(n_splits=4).split(Xw[dev_idx], y[dev_idx], groups[dev_idx]))
    inner_train, inner_val = dev_idx[a], dev_idx[b]
    assert set(groups[inner_train]).isdisjoint(groups[inner_val])
    p_inner = gait_pipeline('P-ALL9', alpha=1.0, n_components=5)
    p_inner.fit(Xw[inner_train], y[inner_train])
    assert p_inner['scale'].n_samples_seen_ == len(inner_train)
    assert p_inner['pca'].n_samples_ == len(inner_train)
    np.testing.assert_allclose(p_inner['scale'].mean_, Xw[inner_train].mean(axis=0),
                               rtol=1e-12, atol=1e-12)
    assert len(p_inner.predict(Xw[inner_val])) == len(inner_val)
    # Refit from precisely the same development trials: held-out rows cannot affect fit.
    X_modified = Xw.copy()
    X_modified[test] += 1e5
    p1 = gait_pipeline('P-ALL9', alpha=1.0, n_components=5).fit(Xw[train], y[train])
    p2 = gait_pipeline('P-ALL9', alpha=1.0, n_components=5).fit(X_modified[train], y[train])
    np.testing.assert_array_equal(p1['scale'].mean_, p2['scale'].mean_)
    np.testing.assert_array_equal(p1['pca'].components_, p2['pca'].components_)
    np.testing.assert_array_equal(p1['ridge'].coef_, p2['ridge'].coef_)
    # The held-out observations were never used to fit any transform.
    assert not np.allclose(p1['scale'].mean_, Xw.mean(axis=0))


def test_pooled_scoring_and_prediction_coverage():
    # Unequal-sized validation folds: pooled trial RMSE != mean of fold RMSEs.
    val_y = np.zeros(4)
    val_pred = np.array([0., 2., 2., 2.])
    np.testing.assert_allclose(pooled_rmse(val_y, val_pred), np.sqrt(3), rtol=0, atol=1e-15)
    assert pooled_rmse(val_y, val_pred) != (0 + 2)/2
    held_out = OUTER_FOLDS[1]
    dev = np.where(~np.isin(groups, held_out))[0]
    test = np.where(np.isin(groups, held_out))[0]
    predictions = {}
    for inner_train, inner_val in GroupKFold(4).split(Xd[dev], y[dev], groups[dev]):
        a, b = dev[inner_train], dev[inner_val]
        model = gait_pipeline('D-ALL9', alpha=1).fit(Xd[a], y[a])
        for index, prediction in zip(b, model.predict(Xd[b])):
            uid = trial_ids[index]
            assert uid not in predictions
            predictions[uid] = float(prediction)
    assert set(predictions) == set(trial_ids[dev])
    outer_model = gait_pipeline('D-ALL9', alpha=1).fit(Xd[dev], y[dev])
    assert len(outer_model.predict(Xd[test])) == len(test)
    assert set(groups[dev]).isdisjoint(groups[test])


def test_contract_whitelist():
    blocks = {'C': np.array([0., 0., 1.]), 'B': np.array([25., 1.8]),
              'G': np.array([3., 4.]),
              'subject_id': np.array([999.]), 'target': np.array([9999.])}
    np.testing.assert_array_equal(assemble_contract('G', blocks), [3, 4])
    np.testing.assert_array_equal(assemble_contract('C', blocks), [0, 0, 1])
    np.testing.assert_array_equal(assemble_contract('C+G', blocks), [0, 0, 1, 3, 4])
    np.testing.assert_array_equal(assemble_contract('C+B', blocks), [0, 0, 1, 25, 1.8])
    np.testing.assert_array_equal(assemble_contract('C+B+G', blocks), [0, 0, 1, 25, 1.8, 3, 4])
    try:
        assemble_contract('C+target', blocks)
    except ValueError:
        pass
    else:
        raise AssertionError('Unknown predictor contract was accepted.')


check('36 frozen W/D/P candidates and deterministic exact ties', test_candidates_and_exact_ties)
check('Five fixed outer participant boundaries and twenty inner group splits', test_outer_inner_group_boundaries)
check('Training-only scaling and PCA for W/D/P, all four PCA counts', test_train_only_preprocessing_and_PCA)
check('Pooled inner trial-level RMSE and one prediction per validation trial', test_pooled_scoring_and_prediction_coverage)
check('Predictor-contract block access excludes labels and participant IDs', test_contract_whitelist)


In [ ]:
# 4 — Archive verification evidence (even if a check failed)
OUT.mkdir(parents=True, exist_ok=True)
with (OUT / 'synthetic_verification_checks.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['check', 'status', 'detail'])
    writer.writeheader()
    writer.writerows(results)

with (OUT / 'synthetic_inner_fold_manifest.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'outer_fold', 'inner_fold', 'train_participants', 'validation_participants',
        'outer_test_participants', 'n_train_trials_synthetic',
        'n_validation_trials_synthetic'])
    writer.writeheader()
    writer.writerows(inner_manifest)

n_pass = sum(r['status'] == 'PASS' for r in results)
summary = {
    'verification_status': 'PASS' if n_pass == len(results) else 'REVIEW_REQUIRED',
    'checks_passed': n_pass, 'checks_total': len(results),
    'uses_real_bacek_data': False, 'trains_real_models': False,
    'uses_gpu': False,
    'synthetic_trial_count': len(groups),
    'synthetic_participant_count': len(participants),
    'fixed_outer_participant_groups': {str(k): list(v) for k, v in OUTER_FOLDS.items()},
    'ridge_alpha_grid': list(ALPHAS), 'pca_component_grid': list(PCA_COUNTS),
    'software': {
        'python': sys.version.split()[0], 'numpy': np.__version__,
        'scikit_learn': sklearn.__version__, 'os': platform.system(),
    },
    'verification_scope': 'Notebook-defined synthetic functions and sklearn split/fitting checks; '
                          'integrated future experimental code requires retesting.',
}
(OUT / 'verification_summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
archive = Path(str(OUT) + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUT.iterdir()):
        z.write(file, arcname=file.name)

print(f'\nVerification: {summary["verification_status"]} ({n_pass}/{len(results)} checks)')
print(f'Results directory: {OUT}')
print(f'Results ZIP: {archive}')
if summary['verification_status'] != 'PASS':
    raise RuntimeError('One or more synthetic verification checks failed; inspect the ZIP.')
